<h4 align="right">By <a href="http://cse.iitkgp.ac.in/~adas/">Abir Das</a> </h4>

## Preamble

To run and solve this tutorial, one must have a working IPython Notebook installation. The easiest way to to run it in Google Colab. Use `Python 3` version. Below statements assume that you have already followed these instructions. If you are new to Python or its scientific library, Numpy, there are some nice tutorials [here](https://www.learnpython.org/) and [here](http://www.scipy-lectures.org/).

In [ ]:
# Import the libraries
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def generate_separable_data(num_points=40, gamma=1.0, beta=10.0, line_angle_deg=45.0, seed=42):
    """
    Generates linearly separable 2D data around a line passing through the origin.
    - gamma: minimum perpendicular distance from the separating line (margin)
    - beta: maximum distance from the origin
    - line_angle_deg: inclination angle of the separating line in degrees
    """
    np.random.seed(seed)

    # Unit normal vector to the separating line
    theta = np.deg2rad(line_angle_deg)
    normal = np.array([np.sin(theta), -np.cos(theta)])
    tangent = np.array([np.cos(theta), np.sin(theta)])

    X = []
    y = []

    # Half positive, half negative
    points_per_class = num_points // 2

    for label in [+1, -1]:
        count = 0
        while count < points_per_class:
            # 1. Perpendicular distance u >= gamma
            u = np.random.uniform(gamma, beta) * label

            # 2. Maximum allowable tangential distance v so that total distance <= beta
            max_v = np.sqrt(max(0.0, beta**2 - u**2))
            v = np.random.uniform(-max_v, max_v)

            # 3. Coordinate reconstruction: x = u * normal + v * tangent
            pt = u * normal + v * tangent

            X.append(pt)
            y.append(label)
            count += 1

    X = np.array(X, dtype='float32')
    y = np.array(y, dtype='float32')

    # Shuffle so classes are interleaved
    shuffle_idx = np.random.permutation(num_points)
    return X[shuffle_idx], y[shuffle_idx]

In [ ]:
gamma = 1   # Margin
beta = 10.0   # Max length from origin
X, y = generate_separable_data(num_points=40, gamma=gamma, beta=beta, line_angle_deg=45.0)

print(f"Generated X shape: {X.shape}, y shape: {y.shape}")
print(f"Theoretical max updates bound (beta^2 / gamma^2): {(beta/gamma)**2:.0f}")

In [ ]:
indexes_with_class1 = y==1
plt.figure(figsize=(8,8))
plt.scatter(X[indexes_with_class1][:,0],X[indexes_with_class1][:,1], color='r', marker='P')
plt.scatter(X[~indexes_with_class1][:,0],X[~indexes_with_class1][:,1], color='g', marker='o')
plt.grid()

# weightplot
plt.xlim(-beta,beta)
plt.ylim(-beta,beta)
plt.show()

In [ ]:
# Initialize weight
w = np.array([0.0, 0.0])
# max_epochs: safety limit to prevent infinite loop if not linearly separable
max_epochs=1000
# Keep track of how many weight updates are happening
update_count = 0
# while no examples are misclassified run the perceptron algorithm (refer to the slides for the algorithm)
for epoch in range(max_epochs):
  misclassified = False

  # If any example is indeed misclassified, then set misclassified = True
  for i in range(X.shape[0]):
    # y_hat = np.dot(w, X[i])
    y_hat = 1.0 if np.dot(w, X[i]) >= 0 else -1.0
    if y_hat != y[i]:
      misclassified = True
      # update weight
      w += y[i]*X[i]
      update_count += 1

  if misclassified == False:
    break

print("Convergence after {:d} weight updates".format(update_count))
# Finally w contains the trained weights

### Scatter plot the data points and plot the straight line given by the trained weights w

In [ ]:
epsilon=1e-18
indexes_with_class1 = y==1
plt.figure(figsize=(10,10))
plt.scatter(X[indexes_with_class1][:,0],X[indexes_with_class1][:,1], color='r', marker='P')
plt.scatter(X[~indexes_with_class1][:,0],X[~indexes_with_class1][:,1], color='g', marker='o')
plt.grid()

# weightplot
plt.xlim(-beta,beta)
plt.ylim(-beta,beta)
slope = (-w[0])/(w[1]+epsilon) # to ensure no division by zero
xx = np.linspace(-15, 15)
yy = slope * xx
plt.plot(xx,yy,color='k', linestyle='-', linewidth=2)
plt.show()